Complete the exercises below For **Assignment #13**.

Load the `ISLR2` and the `tidymodels` packages.

In [17]:
library('tidymodels')
library('ISLR2')

In this assignment we will use the `Default` dataset which includes the default status for credit card customers (`default` variable) in addition to each customer's:

1. credit card balance (`balance` variable),
1. student status (`student` variable), and,
1. income (`income` variable).

In [18]:
Default |> head()

,default,student,balance,income
,<fct>,<fct>,<dbl>,<dbl>
1,No,No,729.5265,44361.625
2,No,Yes,817.1804,12106.135
3,No,No,1073.5492,31767.139
4,No,No,529.2506,35704.494
5,No,No,785.6559,38463.496
6,No,Yes,919.5885,7491.559


We will be modeling `default` with the customer features.

Before we begin let's count how many customers fall into each `default` category.

In [19]:
Default |> count(default)

default,n
<fct>,<int>
No,9667
Yes,333


The data is quite imbalanced. This will be important to keep in mind when we evaluate the performance of our model later. 

Run the code below to create and training data from `Default`. We will use the "test" dataset at the end to get a final evaluation of our best model's accuracy.

In [20]:
Default_split = initial_split(Default, prop = 0.90, strata = default)

Default_train = training(Default_split)
Default_test = testing(Default_split)

Create a logistic regression model called `mod`. Set the engine to `glm` and the mode to `classification`. 

In [21]:
mod=logistic_reg(mode='classification')|>set_engine('glm')

Our data is imbalanced. As such, a naive model that *always* predicts a customer to **not default** would be correct quite often. Let's start by calculating the "accuracy" of a naive model. This will be the baseline accuracy by which we evaluate other models.

In [22]:
# This code calculates the accuracy of a model that always predicts default to be "No"

Default_train |>
    mutate(.pred_naive = factor('No', levels = c('No', 'Yes'))) |>
    accuracy(truth = default, .pred_naive)

.metric,.estimator,.estimate
<chr>,<chr>,<dbl>
accuracy,binary,0.9668889


Let's use k-fold cross validation to evaluate the performance of a model where the outcome is `default` and the predictors are `income` and `balance`.

To start, use `vfold_cv` to generate 10 validation folds (i.e. set the `v` variable to 10). Set the `strata` argument to `default` so we preserve the distribution of `default` values in each fold.

Creat your folds below and use `glimpse` to look at the output table. Call your output folds tables "folds".

In [23]:

folds=vfold_cv(Default, v=10)
folds |> glimpse()

Rows: 10
Columns: 2
$ splits <list> [<vfold_split[9000 x 1000 x 10000 x 4]>], [<vfold_split[9000 x…
$ id     <chr> "Fold01", "Fold02", "Fold03", "Fold04", "Fold05", "Fold06", "Fo…


The code below fits a model to each of your 10 folds. `collect_metrics` finds the average of evaluation metrics for each of your ten models. 

In [34]:
mod |> 
    fit_resamples(default ~ income + balance, folds) |>
    collect_metrics()


.metric,.estimator,mean,n,std_err,.config
<chr>,<chr>,<dbl>,<int>,<dbl>,<chr>
accuracy,binary,0.97370000,10,0.0014533486,pre0_mod0_post0
brier_class,binary,0.02145459,10,0.0008706189,pre0_mod0_post0
roc_auc,binary,0.94919807,10,0.0055485829,pre0_mod0_post0


❓How does the model accuracy compare to the naive model from above?


**The accuracy(0.9737) of current model is slightly higher than the naive model(0.9668889) from above. However, the improvement is small, and given the class imbalance, accuracy alone may not be sufficient to evaluate model performance.**


Complete the cell below to evaluate a model also includes the `student` variable as as predictor.
1. use `default ~ income + balance + student` as the formula,
2. encode your `student` variable with `step_dummy`, and,
3. don't forget to `prep` your recipe!

In [35]:
rec = recipe(default ~ income + balance + student, data=Default)|>
    step_dummy(student)
###seem like we can't do prep() here as it turned it into a trained recipe somehow and trigger an '! Can't add a trained recipe to a workflow.' error

mod |>
    fit_resamples(rec, folds) |>
    collect_metrics()


.metric,.estimator,mean,n,std_err,.config
<chr>,<chr>,<dbl>,<int>,<dbl>,<chr>
accuracy,binary,0.97320000,10,0.0014817407,pre0_mod0_post0
brier_class,binary,0.02141513,10,0.0008356035,pre0_mod0_post0
roc_auc,binary,0.94942304,10,0.0053257847,pre0_mod0_post0


❓Does it appear that the model that includes `student` improves upon the first model with only `income` and `balance` as predictors?

**No.It didn't improve the model performance.**



Finally, estimate the accuracy of an `default ~ income + balance` model on the test data, `Default_test`. 

❓Does our model outperform a naive model?

In [37]:

mod_fit=mod|>fit(default ~ income + balance, data=Default_train)
pred_test=predict(mod_fit, new_data=Default_test)
bind_cols(Default_test, pred_test)|>
    accuracy(truth=default, estimate=.pred_class)


.metric,.estimator,.estimate
<chr>,<chr>,<dbl>
accuracy,binary,0.975



**The model slightly(tiny) out performance the naive model. However, the improvement is small, and given the class imbalance, accuracy alone may not be sufficient to evaluate model performance.**
